# MCD64A1 → 0.25° monthly burned fraction for Iberia

A step-by-step pipeline for building a monthly burned-fraction time series on a 32 × 54 0.25° grid covering the Iberian Peninsula, from MODIS MCD64A1 V061 burned-area data.

**Pipeline.**
1. Download MCD64A1 monthly HDFs for tiles **h17v04**, **h17v05**, **h18v04**, **h18v05**.
2. For each month and tile, project pixel centroids from MODIS sinusoidal to lon/lat and bin into the 0.25° grid.
3. Compute `burned_fraction = burned_pixels / valid_mapped_pixels` per cell.
4. Save the full record as a single NetCDF.

The notebook is structured so each stage can be sanity-checked before running the full ~300-month aggregation.

## Setup

**Requirements:**

GDAL must be built with HDF4 support — verify with `gdalinfo --formats | grep -i HDF4`.

**Authentication:** It checks for `~/.netrc` with credentials for `urs.earthdata.nasa.gov`.

In [ ]:
from pathlib import Path
from datetime import date

import numpy as np
import xarray as xr
import rasterio
import earthaccess
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from osgeo import gdal
gdal.UseExceptions()

## 1. Configuration

The target grid is **cell-centered**: cell centers run from -10.0° to 3.25° E and from 36.25° to 44.0° N in 0.25° steps, giving a 32 × 54 grid (1,728 cells).

MCD64A1 V061 begins November 2000.

The four MODIS sinusoidal tiles spanning Iberia are h17v04, h17v05, h18v04, h18v05 — at lat 40°N the h17/h18 boundary sits at exactly 0° lon, so eastern Spain falls in h18.

In [ ]:
# Target grid
LON_MIN, LON_MAX = -10.0, 3.25
LAT_MIN, LAT_MAX = 36.25, 44.0
DLON = DLAT = 0.25
NLON = int(round((LON_MAX - LON_MIN) / DLON)) + 1   # 54
NLAT = int(round((LAT_MAX - LAT_MIN) / DLAT)) + 1   # 32
LONS = LON_MIN + DLON * np.arange(NLON)
LATS = LAT_MIN + DLAT * np.arange(NLAT)

# Time window
START = date(2000, 11, 1)
END   = date(2025, 12, 1)

# Tiles
TILES = [(17, 4), (17, 5), (18, 4), (18, 5)]

# MODIS sinusoidal grid constants
R_MODIS      = 6371007.181
TILE_SIZE_M  = 1111950.519667
GRID_X_MIN   = -20015109.354
GRID_Y_MAX   =  10007554.677
TILE_PIXELS  = 2400
PIXEL_SIZE_M = TILE_SIZE_M / TILE_PIXELS

# Paths
ROOT       = Path("./data/mcd64a1")
HDF_DIR    = ROOT / "hdf"
CACHE_DIR  = ROOT / "monthly_npz"
NETCDF_OUT = ROOT / "burned_fraction_iberia_0p25deg_monthly.nc"
HDF_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Grid:  {NLAT} lat × {NLON} lon")
print(f"Time:  {START} to {END}")
print(f"Tiles: {TILES}")

## 2. Tile geometry → target-grid indexing

MCD64A1 pixels live on the MODIS sinusoidal grid (500 m). To bin them into our 0.25° lat/lon grid, for each pixel we:

1. Compute its centroid in sinusoidal `(x, y)`.
2. Apply the inverse sinusoidal projection to get `(lon, lat)`.
3. Convert `(lon, lat)` to a target cell index `(i, j)`.

Inverse sinusoidal is just `lat = y / R`, `lon = x / (R · cos(lat))`.

We precompute `(i, j)` for each tile **once** — tile geometry doesn't change month-to-month.

In [ ]:
def tile_pixel_lonlat(h, v):
    '''Pixel-centroid lon/lat for a 2400 x 2400 MODIS tile (h, v).'''
    x_min = GRID_X_MIN + h * TILE_SIZE_M
    y_max = GRID_Y_MAX - v * TILE_SIZE_M
    cols = np.arange(TILE_PIXELS)
    rows = np.arange(TILE_PIXELS)
    x = x_min + (cols + 0.5) * PIXEL_SIZE_M
    y = y_max - (rows + 0.5) * PIXEL_SIZE_M
    X, Y = np.meshgrid(x, y)
    lat_rad = Y / R_MODIS
    lon_rad = X / (R_MODIS * np.cos(lat_rad))
    return np.rad2deg(lon_rad), np.rad2deg(lat_rad)


def precompute_tile_indexing():
    '''For each tile, store (in_bounds_mask, flat_target_index).'''
    indexing = {}
    for (h, v) in TILES:
        lon, lat = tile_pixel_lonlat(h, v)
        j = np.floor((lon - (LON_MIN - DLON / 2)) / DLON).astype(np.int32)
        i = np.floor((lat - (LAT_MIN - DLAT / 2)) / DLAT).astype(np.int32)
        in_bounds = (j >= 0) & (j < NLON) & (i >= 0) & (i < NLAT)
        in_bounds_flat = in_bounds.ravel()
        flat_idx = (i * NLON + j).ravel()[in_bounds_flat].astype(np.int64)
        indexing[(h, v)] = (in_bounds_flat, flat_idx)
        print(f"  tile h{h:02d}v{v:02d}: "
              f"{in_bounds_flat.sum():>9,} / {TILE_PIXELS**2:,} pixels in bbox")
    return indexing

indexing = precompute_tile_indexing()

## 3. Download MCD64A1 HDFs

Downloads 1208 HDF files (~2.5GB) covering Iberia for 2000–2025. **First run takes around five minutes** depending on connection. Re-runs skip files already on disk.

In [ ]:
def download_all():
    earthaccess.login(strategy="netrc")
    results = earthaccess.search_data(
        short_name="MCD64A1",
        version="061",                # <-- was "6.1"
        temporal=(START.isoformat(), END.isoformat()),
        bounding_box=(LON_MIN - 1.5, LAT_MIN - 1.5,
                      LON_MAX + 1.5, LAT_MAX + 1.5),
        count=-1,                     # explicit: return all matches, not the default cap
    )
    print(f"earthaccess: {len(results)} granules; downloading to {HDF_DIR}")
    earthaccess.download(results, local_path=str(HDF_DIR))

download_all()

## 4. Read one HDF (sanity check)

Before processing the full record, read one tile to confirm the HDF reader works and the BurnDate band looks sensible.

We use **July 2017, h17v04** (central Portugal during the Pedrógão Grande complex).

If `rasterio.open` complains about the subdataset name, run `gdalinfo path/to/one.hdf` and confirm the grid name — it should be `MOD_Grid_Monthly_500m_DB_BA` for V061.

In [ ]:
def find_hdf(year, month, h, v):
    '''Locate the MCD64A1 HDF for (year, month, tile).'''
    doy = (date(year, month, 1) - date(year, 1, 1)).days + 1
    pattern = f"MCD64A1.A{year:04d}{doy:03d}.h{h:02d}v{v:02d}.061.*.hdf"
    matches = sorted(HDF_DIR.glob(pattern))
    return matches[-1] if matches else None


def read_burn_date(hdf_path):
    ds = gdal.Open(str(hdf_path))
    # First subdataset is Burn Date (we saw this in the diagnostic output).
    # Filtering by name in case ordering ever changes.
    target = next(name for name, _ in ds.GetSubDatasets()
                  if name.endswith(':"Burn Date"'))
    return gdal.Open(target).ReadAsArray()   # int16, (2400, 2400)

In [ ]:
# Try reading July 2017, h17v04
test_path = find_hdf(2017, 7, 17, 4)
print(f"Reading {test_path}")
bd = read_burn_date(test_path)

print(f"shape  = {bd.shape}, dtype = {bd.dtype}")
print(f"min    = {bd.min()},  max = {bd.max()}")
print(f"unique values (first 15): {np.unique(bd)[:15]}")
print(f"burned pixels (BurnDate >  0): {(bd >  0).sum():,}")
print(f"valid  pixels (BurnDate >= 0): {(bd >= 0).sum():,}")
print(f"water  pixels (BurnDate == -2): {(bd == -2).sum():,}")

In [ ]:
# Visualise burned mask on the raw tile grid
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow((bd > 0), cmap="Reds", interpolation="nearest")
ax.set_title("h17v04, July 2017 — burned pixels (BurnDate > 0)")
ax.set_xlabel("column"); ax.set_ylabel("row")
plt.tight_layout(); plt.show()

## 5. Aggregate one month (sanity check)

Bin all four tiles into the 0.25° grid for **June 2017** — the Pedrógão Grande month — and plot the burned fraction. Expect a clear signal in central Portugal.

In [ ]:
def aggregate_month(year, month, indexing):
    '''Aggregate one month across all tiles -> (burned, valid) (NLAT, NLON).'''
    burned = np.zeros((NLAT, NLON), dtype=np.float64)
    valid  = np.zeros((NLAT, NLON), dtype=np.float64)
    NCELLS = NLAT * NLON

    for (h, v) in TILES:
        hdf_path = find_hdf(year, month, h, v)
        if hdf_path is None:
            print(f"  [warn] missing tile h{h:02d}v{v:02d}")
            return None
        bd = read_burn_date(hdf_path).ravel()
        in_bounds_flat, flat_idx = indexing[(h, v)]

        bd_in = bd[in_bounds_flat]
        is_burned = (bd_in >  0).astype(np.float64)
        is_valid  = (bd_in >= 0).astype(np.float64)

        burned += np.bincount(flat_idx, weights=is_burned,
                              minlength=NCELLS).reshape(NLAT, NLON)
        valid  += np.bincount(flat_idx, weights=is_valid,
                              minlength=NCELLS).reshape(NLAT, NLON)

    return burned, valid

In [ ]:
# Aggregate June 2017
burned_jun17, valid_jun17 = aggregate_month(2017, 6, indexing)
frac_jun17 = np.where(valid_jun17 > 0, burned_jun17 / valid_jun17, np.nan)

print(f"max burned fraction: {np.nanmax(frac_jun17):.4f}")
print(f"cells > 1% burned:   {(frac_jun17 > 0.01).sum()}")
print(f"cells > 5% burned:   {(frac_jun17 > 0.05).sum()}")

extent = [LON_MIN - DLON/2, LON_MAX + DLON/2,
          LAT_MIN - DLAT/2, LAT_MAX + DLAT/2]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(frac_jun17, extent=extent, origin="lower",
               cmap="hot_r", vmin=0, vmax=0.1)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Burned fraction — June 2017 (Pedrógão Grande)")
plt.colorbar(im, ax=ax, label="Burned fraction")
plt.tight_layout(); plt.show()

## 6. Aggregate all months

If June 2017 looks right above, run the full record. Per-month results are cached to `.npz`, so re-running is fast.

In [ ]:
def process_all_months(indexing):
    '''Aggregate every month in [START, END]. Cached per month.'''
    out = []
    cur = START
    while cur <= END:
        cache_path = CACHE_DIR / f"{cur:%Y-%m}.npz"
        if cache_path.exists():
            with np.load(cache_path) as f:
                out.append((cur, f["burned"], f["valid"]))
        else:
            print(f"Aggregating {cur:%Y-%m}...")
            result = aggregate_month(cur.year, cur.month, indexing)
            if result is not None:
                burned, valid = result
                np.savez_compressed(cache_path, burned=burned, valid=valid)
                out.append((cur, burned, valid))
            else:
                print(f"  -> skipped {cur:%Y-%m} (missing tile)")
        cur += relativedelta(months=1)
    return out

monthly = process_all_months(indexing)
print(f"\nAggregated {len(monthly)} months: {monthly[0][0]} to {monthly[-1][0]}")

## 7. Build NetCDF

Stack all monthly maps into a single `(time, lat, lon)` xarray Dataset and write it as a NetCDF. We keep the raw `burned_pixels` and `valid_pixels` counts alongside the fraction so the denominator definition can be revisited later without re-reading the HDFs.

In [ ]:
def build_netcdf(monthly, path):
    times  = np.array([np.datetime64(d, "ns") for d, _, _ in monthly])
    burned = np.stack([b for _, b, _ in monthly], axis=0)
    valid  = np.stack([v for _, _, v in monthly], axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        frac = np.where(valid > 0, burned / valid, np.nan)

    ds = xr.Dataset(
        data_vars=dict(
            burned_fraction=(("time", "lat", "lon"),
                             frac.astype(np.float32),
                             dict(long_name="Burned fraction", units="1",
                                  valid_range=(0.0, 1.0))),
            burned_pixels=(("time", "lat", "lon"),
                           burned.astype(np.int32),
                           dict(long_name="MODIS pixels burned in month")),
            valid_pixels=(("time", "lat", "lon"),
                          valid.astype(np.int32),
                          dict(long_name="MODIS pixels mapped as land")),
        ),
        coords=dict(
            time=times,
            lat=("lat", LATS, dict(units="degrees_north", standard_name="latitude")),
            lon=("lon", LONS, dict(units="degrees_east",  standard_name="longitude")),
        ),
        attrs=dict(
            title="MCD64A1 monthly burned fraction, Iberia 0.25° grid",
            source="MODIS MCD64A1 V061 (LP DAAC)",
            method="Pixel-centroid binning from sinusoidal to lon/lat",
            burned_definition="BurnDate > 0",
            valid_definition="BurnDate >= 0 (excludes -1 unmapped, -2 water)",
        ),
    )
    encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
    ds.to_netcdf(path, encoding=encoding)
    return ds

ds = build_netcdf(monthly, NETCDF_OUT)
print(f"Wrote {NETCDF_OUT}")
ds

## 8. Final check — 2017 case studies

The two big 2017 fire months in Portugal were **June** (Pedrógão Grande) and **October**. Both should show clear signal.

In [ ]:
ds = xr.open_dataset(NETCDF_OUT)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
extent = [LON_MIN - DLON/2, LON_MAX + DLON/2,
          LAT_MIN - DLAT/2, LAT_MAX + DLAT/2]

for ax, month_str in zip(axes, ["2017-06", "2017-10"]):
    arr = ds["burned_fraction"].sel(time=month_str).values
    if arr.ndim == 3:    # in case .sel returns a slice
        arr = arr[0]
    im = ax.imshow(arr, extent=extent, origin="lower",
                   cmap="hot_r", vmin=0, vmax=0.15)
    ax.set_title(f"Burned fraction — {month_str}")
    ax.set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
fig.colorbar(im, ax=axes, label="Burned fraction", shrink=0.85)
plt.show()

# Bonus: full annual time series of total burned fraction
total = ds["burned_pixels"].sum(("lat", "lon")) / ds["valid_pixels"].sum(("lat", "lon"))
fig, ax = plt.subplots(figsize=(12, 3))
total.plot(ax=ax)
ax.set_title("Iberia-total burned fraction, monthly")
ax.set_ylabel("Fraction")
plt.tight_layout(); plt.show()